# FFmpeg Processing Capability

> FFmpeg-based media processing capability implementing the `MediaProcessingCapability` interface.

In [ ]:
#| default_exp capability

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
import json
import logging
import os
import subprocess
import uuid
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Union

from cjm_substrate.core.capability import ToolCapability
from cjm_substrate.core.errors import CapabilityInputError
from cjm_substrate.utils.validation import (
    dict_to_config, config_to_dict, dataclass_to_jsonschema,
    SCHEMA_TITLE, SCHEMA_DESC, SCHEMA_ENUM
)

# Stage 8 (Option C / PILLAR 1c): the tool re-bases onto ToolCapability (pure
# compute). The cache/persist bookends + the (action,input,config)->output_dir
# choice moved OUT to the generic media-processing adapter
# (cjm-media-processing-adapter-interface); the result DTOs live in
# cjm-capability-primitives. No get_plugin_metadata, no self.storage, no in-tool
# cache_dir_for_config/hashing. The adapter tells each artifact op WHERE to write.
from cjm_capability_primitives.media_processing import (
    MediaArtifactResult, MediaSegmentationResult, MediaSegment, MediaMetadata,
)

# FFmpeg helpers (lifted from the retired cjm-ffmpeg-utils into this capability's utils/).
# Absolute intra-package imports in the NOTEBOOK source — nbdev relativizes them
# in the generated .py (nbdev rejects source-level relative imports; stage-8 L7).
from cjm_capability_ffmpeg.utils.availability import FFMPEG_AVAILABLE
from cjm_capability_ffmpeg.utils.codec import get_audio_codec
from cjm_capability_ffmpeg.utils.probe import get_media_duration
from cjm_capability_ffmpeg.utils.segments import extract_audio_segment
from cjm_capability_ffmpeg.utils.progress import run_ffmpeg_with_progress

## Configuration

In [ ]:
#| export
@dataclass
class FFmpegCapabilityConfig:
    """Configuration for the FFmpeg processing tool (the HOW knobs only).

    Per-call WHAT-to-produce params (output_format / sample_rate / channels /
    boundaries) are method arguments the adapter hashes into the cache key, NOT
    config — only the quality/engine knobs live here. (The fused-era `output_dir`
    config field is gone: the adapter chooses the output location, stage-8 L6.)"""

    default_audio_format: str = field(
        default="mp3",
        metadata={
            SCHEMA_TITLE: "Default Audio Format",
            SCHEMA_DESC: "Fallback format for audio extraction when the source codec has no clean container mapping.",
            SCHEMA_ENUM: ["mp3", "wav", "flac", "aac", "ogg", "m4a"]
        }
    )

    default_audio_bitrate: str = field(
        default="192k",
        metadata={
            SCHEMA_TITLE: "Default Audio Bitrate",
            SCHEMA_DESC: "Bitrate for audio encoding operations (a quality knob; in the adapter cache key via get_current_config).",
            SCHEMA_ENUM: ["128k", "192k", "256k", "320k"]
        }
    )

    prefer_stream_copy: bool = field(
        default=True,
        metadata={
            SCHEMA_TITLE: "Prefer Stream Copy",
            SCHEMA_DESC: "When possible, copy the audio stream without re-encoding (faster, lossless)."
        }
    )

    resampler: str = field(
        default="soxr",
        metadata={
            SCHEMA_TITLE: "Resampler",
            SCHEMA_DESC: "Audio resampling engine for sample-rate conversion. 'soxr' (libsoxr, high quality, matches librosa's soxr_hq) is preferred over ffmpeg's default 'swr'; needs an ffmpeg built --enable-libsoxr.",
            SCHEMA_ENUM: ["soxr", "swr"]
        }
    )

In [ ]:
schema = dataclass_to_jsonschema(FFmpegCapabilityConfig)
assert "properties" in schema
assert "default_audio_format" in schema["properties"]
assert schema["properties"]["default_audio_format"]["default"] == "mp3"
print(f"Config schema: {list(schema['properties'].keys())}")

Config schema: ['output_dir', 'default_audio_format', 'default_audio_bitrate', 'prefer_stream_copy']


## Capability Class

In [ ]:
#| export
class FFmpegProcessingCapability(ToolCapability):
    """FFmpeg-based media-processing tool capability (stage 8: pure compute).

    Native-surface model (PILLAR 1c): PURE COMPUTE. The artifact ops
    (`convert`/`segment_audio`/`extract_audio`) read input, run ffmpeg, and WRITE
    file(s) into the adapter-supplied `output_dir`, returning typed DTOs;
    `get_info` probes and returns `MediaMetadata`. The cache-check +
    output-location choice + persistence live in the generic media-processing
    adapter (cjm-media-processing-adapter-interface); result DTOs live in
    cjm-capability-primitives; identity derives from the installed distribution.
    No `get_plugin_metadata`, no `self.storage`, no in-tool cache, no
    `execute`/`@capability_action` dispatcher (the task channel routes by method).

    Config = the HOW knobs (resampler engine, bitrate, stream-copy preference,
    fallback format); the per-call output_format/sample_rate/channels/boundaries
    are WHAT-to-produce request params the adapter hashes into the cache key.
    The fused-era `extract_segment` action was DROPPED (no consumer; the HITL
    audio-chunk read is a future in-memory op, a different shape)."""

    config_class = FFmpegCapabilityConfig

    def __init__(self):
        """Initialize the FFmpeg processing tool."""
        self.logger = logging.getLogger(f"{__name__}.{type(self).__name__}")
        self.config: Optional[FFmpegCapabilityConfig] = None

    @property
    def name(self) -> str:  # Tool identity, derived from the installed distribution (PILLAR 1c)
        from importlib.metadata import metadata, packages_distributions
        dist = (packages_distributions().get(__package__) or [__package__.replace("_", "-")])[0]
        return metadata(dist)["Name"]

    @property
    def version(self) -> str:  # Tool version
        from cjm_capability_ffmpeg import __version__
        return __version__

    def initialize(self,
                   config: Optional[Any] = None,  # Configuration dict or None for defaults
                  ) -> None:
        """First-time setup. No storage init — the adapter owns the cache (stage 8)."""
        self.config = dict_to_config(FFmpegCapabilityConfig, config or {})
        self.logger.info(f"Initialized FFmpeg tool (format={self.config.default_audio_format})")

    def get_config_schema(self) -> Dict[str, Any]:  # JSON Schema for UI form generation
        """Return the JSON Schema for tool configuration."""
        return dataclass_to_jsonschema(FFmpegCapabilityConfig)

    def get_current_config(self) -> Dict[str, Any]:  # Current config as dict
        """Return the current configuration as a dictionary."""
        return config_to_dict(self.config) if self.config else {}

    def is_available(self) -> bool:  # Whether ffmpeg is installed
        """Check if ffmpeg is available on this system."""
        return FFMPEG_AVAILABLE

    # ------------------------------------------------------------------
    # Private helper
    # ------------------------------------------------------------------

    def _detect_audio_codec(self,
                            file_path: str,  # Path to media file
                           ) -> Optional[str]:  # Audio codec name or None
        """Detect the audio codec in a media file via ffprobe."""
        cmd = [
            'ffprobe', '-v', 'quiet',
            '-select_streams', 'a:0',
            '-show_entries', 'stream=codec_name',
            '-of', 'csv=p=0',
            file_path
        ]
        try:
            result = subprocess.run(cmd, check=True, capture_output=True, text=True)
            codec = result.stdout.strip()
            return codec if codec else None
        except subprocess.CalledProcessError:
            return None

    # ------------------------------------------------------------------
    # Pure-compute probe (uncached)
    # ------------------------------------------------------------------

    def get_info(self,
                 file_path: Union[str, Path],  # Path to media file
                ) -> MediaMetadata:  # Probed metadata
        """Probe metadata for a media file via ffprobe (UNCACHED pure-query)."""
        file_path = str(file_path)
        if not os.path.exists(file_path):
            raise CapabilityInputError(  # SG-47: missing input file
                f"File not found: {file_path}", fields_invalid=["file_path"],
            )

        cmd = [
            'ffprobe', '-v', 'quiet',
            '-show_format', '-show_streams', '-of', 'json',
            file_path
        ]
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        probe_data = json.loads(result.stdout)

        format_info = probe_data.get('format', {})
        duration = float(format_info.get('duration', 0))
        size_bytes = int(format_info.get('size', 0))
        container_format = format_info.get('format_name', '')

        video_streams = []
        audio_streams = []
        for stream in probe_data.get('streams', []):
            codec_type = stream.get('codec_type', '')
            if codec_type == 'video':
                video_streams.append({
                    'codec': stream.get('codec_name'),
                    'width': stream.get('width'),
                    'height': stream.get('height'),
                    'fps': stream.get('r_frame_rate'),
                })
            elif codec_type == 'audio':
                stream_duration = float(stream.get('duration', duration))
                audio_streams.append({
                    'codec': stream.get('codec_name'),
                    'sample_rate': int(stream.get('sample_rate', 0)),
                    'channels': int(stream.get('channels', 0)),
                    'duration': stream_duration,
                })

        return MediaMetadata(
            path=file_path,
            duration=duration,
            format=container_format,
            size_bytes=size_bytes,
            video_streams=video_streams,
            audio_streams=audio_streams,
        )

    # ------------------------------------------------------------------
    # Pure-compute artifact ops (write into the adapter-supplied output_dir)
    # ------------------------------------------------------------------

    def convert(self,
                input_path: Union[str, Path],     # Source (or upstream artifact) media to convert
                output_dir: str,                  # Adapter-chosen dir to write the artifact into
                output_format: str,               # Target audio format (e.g. 'wav', 'mp3')
                sample_rate: Optional[int] = None,  # Target sample rate (e.g. 16000); None = source rate
                channels: Optional[int] = None,     # Target channel count (e.g. 1 = mono); None = source
                **kwargs                          # Provenance pass-through (unused by compute)
               ) -> MediaArtifactResult:  # The produced converted-audio artifact
        """Convert media to target/model-ready audio — PURE COMPUTE.

        Writes `<output_dir>/<stem>.<output_format>` and returns its pointer. The
        quality knobs (bitrate, resampler) come from `self.config`; the shape
        params (format/rate/channels) are the per-call request (the adapter hashes
        them into the cache key)."""
        input_path = str(input_path)
        if not os.path.exists(input_path):
            raise CapabilityInputError(  # SG-47: missing input file
                f"File not found: {input_path}", fields_invalid=["input_path"],
            )

        bitrate = self.config.default_audio_bitrate
        resampler = self.config.resampler

        os.makedirs(output_dir, exist_ok=True)
        stem = Path(input_path).stem
        output_path = os.path.join(output_dir, f"{stem}.{output_format}")

        total_duration = get_media_duration(Path(input_path))
        codec = get_audio_codec(output_format)

        cmd = ['ffmpeg', '-i', input_path, '-vn', '-acodec', codec]
        if bitrate:
            cmd.extend(['-b:a', bitrate])
        if sample_rate:
            # Resample with the configured engine (soxr by default — libsoxr at HQ
            # precision, matching librosa's soxr_hq), standardizing model-ready input.
            if resampler:
                cmd.extend(['-af', f'aresample=resampler={resampler}:precision=20'])
            cmd.extend(['-ar', str(sample_rate)])
        if channels:
            cmd.extend(['-ac', str(channels)])
        cmd.extend(['-progress', 'pipe:2', '-y', output_path])

        self.report_progress(0.0, f"Converting to {output_format}...")
        run_ffmpeg_with_progress(
            cmd=cmd, total_duration=total_duration,
            description=f"Converting to {output_format}"
        )
        self.report_progress(1.0, "Complete")
        self.logger.info(f"Converted {input_path} -> {output_path}")

        return MediaArtifactResult(output_path=output_path, metadata={
            "output_format": output_format, "sample_rate": sample_rate, "channels": channels,
            "resampler": resampler, "bitrate": bitrate, "duration": total_duration,
        })

    def segment_audio(self,
                      input_path: Union[str, Path],         # Source audio to cut
                      output_dir: str,                      # Adapter-chosen dir to write the segments into
                      boundaries: List[Dict[str, float]],   # [{"start", "end"}, ...]
                      output_format: Optional[str] = None,  # Output format (default: same as input)
                      filename_template: str = "segment_{index:03d}",  # Per-segment filename pattern
                      **kwargs                              # Provenance pass-through (unused by compute)
                     ) -> MediaSegmentationResult:  # The produced batch of segment files
        """Split an audio file into segments at the given boundaries — PURE COMPUTE.

        Writes one file per boundary into `output_dir` and returns the typed batch
        result. Batch-level caching is the adapter's concern (one row per
        (input, boundaries+format config); the per-segment resume-skip the fused
        tool did is intentionally not reproduced)."""
        input_path = str(input_path)
        if not os.path.exists(input_path):
            raise CapabilityInputError(  # SG-47: missing input file
                f"File not found: {input_path}", fields_invalid=["input_path"],
            )
        if not boundaries:
            raise CapabilityInputError(  # SG-47: typed input-validation
                "boundaries list must not be empty", fields_invalid=["boundaries"],
            )
        # Validate boundaries are sorted and non-overlapping.
        for i, b in enumerate(boundaries):
            if b["end"] <= b["start"]:
                raise CapabilityInputError(
                    f"Boundary {i}: end ({b['end']}) must be > start ({b['start']})",
                    fields_invalid=["boundaries"],
                )
            if i > 0 and b["start"] < boundaries[i - 1]["end"]:
                raise CapabilityInputError(
                    f"Boundary {i} start ({b['start']}) overlaps boundary {i-1} end ({boundaries[i-1]['end']})",
                    fields_invalid=["boundaries"],
                )

        ext = output_format or Path(input_path).suffix.lstrip('.')
        os.makedirs(output_dir, exist_ok=True)
        batch_key = str(uuid.uuid4())
        total = len(boundaries)

        segments: List[MediaSegment] = []
        for i, boundary in enumerate(boundaries):
            start = boundary["start"]
            end = boundary["end"]
            duration = end - start
            filename = f"{filename_template.format(index=i)}.{ext}"
            output_path = os.path.join(output_dir, filename)
            extract_audio_segment(
                input_path=Path(input_path),
                output_path=Path(output_path),
                start_time=str(start),
                duration=str(duration),
            )
            segments.append(MediaSegment(
                index=i, output_path=output_path, start=start, end=end, duration=duration,
            ))
            self.report_progress((i + 1) / total, f"Segment {i + 1}/{total}")

        total_duration = sum(s.duration for s in segments)
        self.report_progress(1.0, f"Complete: {total} segments")
        self.logger.info(f"Segmented {input_path} into {total} segments (batch_key={batch_key})")

        return MediaSegmentationResult(
            segments=segments, input_path=input_path, segment_count=total,
            total_duration=total_duration, batch_key=batch_key,
        )

    def extract_audio(self,
                      input_path: Union[str, Path],         # Source video container
                      output_dir: str,                      # Adapter-chosen dir to write the artifact into
                      output_format: Optional[str] = None,  # Audio format (default: from codec detection)
                      **kwargs                              # Provenance pass-through (unused by compute)
                     ) -> MediaArtifactResult:  # The produced extracted-audio artifact
        """Extract the audio stream from a video container — PURE COMPUTE.

        Stream-copies when the source codec maps cleanly (lossless, fast),
        otherwise re-encodes to the resolved format. Writes into `output_dir`."""
        input_path = str(input_path)
        if not os.path.exists(input_path):
            raise CapabilityInputError(  # SG-47: missing input file
                f"File not found: {input_path}", fields_invalid=["input_path"],
            )

        codec = self._detect_audio_codec(input_path)
        if not codec:
            raise CapabilityInputError(  # SG-47: input file has no audio stream
                f"No audio stream found in: {input_path}", fields_invalid=["input_path"],
            )

        stream_copy = self.config.prefer_stream_copy
        if output_format is None:
            codec_ext_map = {
                'aac': 'm4a', 'mp3': 'mp3', 'vorbis': 'ogg', 'opus': 'ogg',
                'flac': 'flac', 'pcm_s16le': 'wav', 'pcm_s24le': 'wav',
            }
            output_format = codec_ext_map.get(codec, self.config.default_audio_format)
            if codec not in codec_ext_map:
                stream_copy = False
        else:
            expected_codec = get_audio_codec(output_format)
            if expected_codec != 'copy' and expected_codec != codec:
                stream_copy = False

        os.makedirs(output_dir, exist_ok=True)
        stem = Path(input_path).stem
        output_path = os.path.join(output_dir, f"{stem}.{output_format}")
        total_duration = get_media_duration(Path(input_path))

        cmd = ['ffmpeg', '-i', input_path, '-vn']
        if stream_copy:
            cmd.extend(['-acodec', 'copy'])
        else:
            cmd.extend(['-acodec', get_audio_codec(output_format)])
            cmd.extend(['-b:a', self.config.default_audio_bitrate])
        cmd.extend(['-progress', 'pipe:2', '-y', output_path])

        self.report_progress(0.1, "Extracting audio stream...")
        run_ffmpeg_with_progress(
            cmd=cmd, total_duration=total_duration,
            description="Extracting audio"
        )
        self.report_progress(1.0, "Complete")
        self.logger.info(f"Extracted audio {input_path} -> {output_path} (stream_copy={stream_copy})")

        return MediaArtifactResult(output_path=output_path, metadata={
            "codec": codec, "duration": total_duration, "output_format": output_format,
            "stream_copy": stream_copy, "source_video_path": input_path,
        })

In [ ]:
capability = FFmpegProcessingCapability()
# `name` derives from importlib.metadata and is unrunnable in the notebook __main__
# cell (__package__ is None) — asserted at manifest-gen / e2e (the silero/Demucs
# precedent). Here: version + availability + the native surface.
assert isinstance(capability.version, str) and capability.version
assert capability.is_available() == FFMPEG_AVAILABLE
print(f"FFmpeg tool v{capability.version}; available: {capability.is_available()}")

In [ ]:
capability.initialize({})
assert capability.config is not None
assert capability.config.default_audio_format == "mp3"
assert capability.config.prefer_stream_copy is True
assert capability.config.resampler == "soxr"
# No self.storage anymore — the adapter owns the cache (stage 8).
assert not hasattr(capability, "storage")
print(f"Config: format={capability.config.default_audio_format}, bitrate={capability.config.default_audio_bitrate}, resampler={capability.config.resampler}")
print(f"Current config: {capability.get_current_config()}")

In [ ]:
capability.initialize({"default_audio_format": "wav", "default_audio_bitrate": "320k", "resampler": "swr"})
assert capability.config.default_audio_format == "wav"
assert capability.config.default_audio_bitrate == "320k"
assert capability.config.resampler == "swr"
print(f"Custom config: {capability.get_current_config()}")

In [ ]:
schema = capability.get_config_schema()
assert "properties" in schema
props = schema["properties"]
# The fused-era `output_dir` config field is GONE (the adapter chooses output location).
assert "output_dir" not in props
assert "default_audio_format" in props
assert "default_audio_bitrate" in props
assert "prefer_stream_copy" in props
assert "resampler" in props
print(f"Schema properties: {list(props.keys())}")

In [ ]:
# Self-contained instance (avoid cross-cell state for nbdev-readme/quarto parallel exec)
capability = FFmpegProcessingCapability()
# Native-surface assertion: the pure-compute tool exposes the media-processing
# methods + get_current_config (the MediaProcessingToolProtocol surface) and NO
# fused-era execute/@capability_action dispatcher or supported_actions.
for m in ("convert", "segment_audio", "extract_audio", "get_info", "get_current_config"):
    assert callable(getattr(capability, m, None)), f"missing native method {m}"
assert not hasattr(capability, "supported_actions"), "fused-era action dispatcher must be gone"
# extract_segment was dropped (no consumer; future HITL read is a distinct in-memory op).
assert not hasattr(capability, "extract_segment")
import inspect
assert "output_dir" in inspect.signature(capability.convert).parameters
print("Native media-processing surface OK (convert/segment_audio/extract_audio/get_info; no execute/extract_segment)")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()